In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [33]:
train_df = pd.read_csv("/kaggle/input/datasets/shreyash28/pharam1/train.csv")
test_df = pd.read_csv("/kaggle/input/datasets/shreyash28/pharam1/test.csv")
sub_df = pd.read_csv("/kaggle/input/datasets/shreyash28/pharam1/sample_submission.csv")

In [34]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/shreyash28/pharam1/train.csv")

# --- Shape ---
print("Shape:", train_df.shape)

# --- Column names + dtypes ---
print("\nColumns & Dtypes:")
print(train_df.dtypes)

# --- First 5 rows ---
print("\nHead:")
display(train_df.head())

# --- Basic stats (numeric) ---
print("\nDescribe (numeric):")
display(train_df.describe())

# --- Nulls ---
print("\nNull counts:")
print(train_df.isnull().sum())

# --- Cardinality of object columns ---
cat_cols = train_df.select_dtypes(include='object').columns
if len(cat_cols) > 0:
    print("\nCategorical column unique value counts:")
    for col in cat_cols:
        print(f"  {col}: {train_df[col].nunique()} unique → {train_df[col].unique()[:5]}")

Shape: (22083, 45)

Columns & Dtypes:
Patient Id                                           object
Patient Age                                         float64
Genes in mother's side                               object
Inherited from father                                object
Maternal gene                                        object
Paternal gene                                        object
Blood cell count (mcL)                              float64
Patient First Name                                   object
Family Name                                          object
Father's name                                        object
Mother's age                                        float64
Father's age                                        float64
Institute Name                                       object
Location of Institute                                object
Status                                               object
Respiratory Rate (breaths/min)                       object
He

,Patient Id,Patient Age,Genes in mother's side,Inherited from father,Maternal gene,Paternal gene,Blood cell count (mcL),Patient First Name,Family Name,Father's name,...,Birth defects,White Blood cell count (thousand per microliter),Blood test result,Symptom 1,Symptom 2,Symptom 3,Symptom 4,Symptom 5,Genetic Disorder,Disorder Subclass
0,PID0x6418,2.0,Yes,No,Yes,No,4.760603,Richard,NaN,Larre,...,NaN,9.857562,NaN,1.0,1.0,1.0,1.0,1.0,Mitochondrial genetic inheritance disorders,Leber's hereditary optic neuropathy
1,PID0x25d5,4.0,Yes,Yes,No,No,4.910669,Mike,NaN,Brycen,...,Multiple,5.522560,normal,1.0,NaN,1.0,1.0,0.0,NaN,Cystic fibrosis
2,PID0x4a82,6.0,Yes,No,No,No,4.893297,Kimberly,NaN,Nashon,...,Singular,NaN,normal,0.0,1.0,1.0,1.0,1.0,Multifactorial genetic inheritance disorders,Diabetes
3,PID0x4ac8,12.0,Yes,No,Yes,No,4.705280,Jeffery,Hoelscher,Aayaan,...,Singular,7.919321,inconclusive,0.0,0.0,1.0,0.0,0.0,Mitochondrial genetic inheritance disorders,Leigh syndrome
4,PID0x1bf7,11.0,Yes,No,NaN,Yes,4.720703,Johanna,Stutzman,Suave,...,Multiple,4.098210,NaN,0.0,0.0,0.0,0.0,NaN,Multifactorial genetic inheritance disorders,Cancer



Describe (numeric):


,Patient Age,Blood cell count (mcL),Mother's age,Father's age,Test 1,Test 2,Test 3,Test 4,Test 5,No. of previous abortion,White Blood cell count (thousand per microliter),Symptom 1,Symptom 2,Symptom 3,Symptom 4,Symptom 5
count,20656.000000,22083.000000,16047.000000,16097.000000,19956.0,19931.0,19936.0,19943.0,19913.0,19921.000000,19935.000000,19928.000000,19861.000000,19982.000000,19970.000000,19930.000000
mean,6.974148,4.898871,34.526454,41.972852,0.0,0.0,0.0,1.0,0.0,2.003062,7.486224,0.592483,0.551886,0.536233,0.497747,0.461917
std,4.319475,0.199663,9.852598,13.035501,0.0,0.0,0.0,0.0,0.0,1.411919,2.653393,0.491385,0.497313,0.498698,0.500007,0.498560
min,0.000000,4.092727,18.000000,20.000000,0.0,0.0,0.0,1.0,0.0,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,4.763109,26.000000,31.000000,0.0,0.0,0.0,1.0,0.0,1.000000,5.424703,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.000000,4.899399,35.000000,42.000000,0.0,0.0,0.0,1.0,0.0,2.000000,7.477132,1.000000,1.000000,1.000000,0.000000,0.000000
75%,11.000000,5.033830,43.000000,53.000000,0.0,0.0,0.0,1.0,0.0,3.000000,9.526152,1.000000,1.000000,1.000000,1.000000,1.000000
max,14.000000,5.609829,51.000000,64.000000,0.0,0.0,0.0,1.0,0.0,4.000000,12.000000,1.000000,1.000000,1.000000,1.000000,1.000000



Null counts:
Patient Id                                             0
Patient Age                                         1427
Genes in mother's side                                 0
Inherited from father                                306
Maternal gene                                       2810
Paternal gene                                          0
Blood cell count (mcL)                                 0
Patient First Name                                     0
Family Name                                         9691
Father's name                                          0
Mother's age                                        6036
Father's age                                        5986
Institute Name                                      5106
Location of Institute                                  0
Status                                                 0
Respiratory Rate (breaths/min)                      2149
Heart Rate (rates/min                               2113
Test 1           

In [35]:
class XGBoostRegressor:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.reg_lambda = reg_lambda
        self.trees = []
        self.base_pred = 0

    def fit(self, X, y):
        self.base_pred = np.mean(y)
        f_m = np.full(y.shape, self.base_pred)

        for _ in range(self.n_estimators):
            # Calculate gradients (g) and hessians (h) for MSE
            g = f_m - y 
            h = np.ones_like(y)
            
            tree = XGBoostTree(max_depth=self.max_depth, reg_lambda=self.reg_lambda)
            tree.tree = tree.build_tree(X, g, h)
            
            # Update predictions: F_m = F_m-1 + lr * tree_pred
            preds = np.array([tree.predict_row(row, tree.tree) for row in X])
            f_m += self.lr * preds 
            self.trees.append(tree)

    def predict(self, X):
        y_pred = np.full(X.shape[0], self.base_pred)
        for tree in self.trees:
            y_pred += self.lr * np.array([tree.predict_row(row, tree.tree) for row in X])
        return y_pred


In [36]:
import numpy as np
import pandas as pd
from copy import deepcopy

from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


class XGBoostRegressor:
    def __init__(
        self,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        early_stopping_rounds=None,
        verbose=False
    ):
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.reg_lambda = reg_lambda
        self.random_state = random_state
        self.early_stopping_rounds = early_stopping_rounds
        self.verbose = verbose

        self.base_pred = 0.0
        self.trees = []
        self.feature_sets = []
        self.preprocessor = None

    def _build_preprocessor(self, X):
        if not isinstance(X, pd.DataFrame):
            return None

        num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = [c for c in X.columns if c not in num_cols]

        transformers = []

        if num_cols:
            transformers.append((
                "num",
                SimpleImputer(strategy="median"),
                num_cols
            ))

        if cat_cols:
            transformers.append((
                "cat",
                PipelineLike([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
                ]),
                cat_cols
            ))

        return ColumnTransformer(transformers, remainder="drop")

    def _to_matrix(self, X, fit=False):
        if isinstance(X, pd.DataFrame):
            if fit:
                self.preprocessor = self._build_preprocessor(X)
                Xp = self.preprocessor.fit_transform(X)
            else:
                Xp = self.preprocessor.transform(X)
            return np.asarray(Xp, dtype=np.float32)

        X = np.asarray(X, dtype=np.float32)
        if np.isnan(X).any():
            col_medians = np.nanmedian(X, axis=0)
            inds = np.where(np.isnan(X))
            X = X.copy()
            X[inds] = np.take(col_medians, inds[1])
        return X

    def fit(self, X, y, eval_set=None):
        y = np.asarray(y, dtype=np.float32).ravel()
        rng = np.random.RandomState(self.random_state)

        Xp = self._to_matrix(X, fit=True)
        n_samples, n_features = Xp.shape

        self.base_pred = float(np.mean(y))
        pred = np.full(n_samples, self.base_pred, dtype=np.float32)

        best_score = np.inf
        best_trees = []
        best_feature_sets = []
        rounds_no_improve = 0

        if eval_set is not None:
            X_val, y_val = eval_set
            y_val = np.asarray(y_val, dtype=np.float32).ravel()
            X_valp = self._to_matrix(X_val, fit=False)

        for m in range(self.n_estimators):
            residual = y - pred

            row_size = max(1, int(self.subsample * n_samples))
            col_size = max(1, int(self.colsample_bytree * n_features))

            rows = rng.choice(n_samples, size=row_size, replace=False)
            cols = rng.choice(n_features, size=col_size, replace=False)

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                random_state=rng.randint(0, 10**9)
            )

            tree.fit(Xp[rows][:, cols], residual[rows])
            update = tree.predict(Xp[:, cols])

            # simple L2 shrinkage via learning rate
            pred += self.lr * update / (1.0 + self.reg_lambda)

            self.trees.append(tree)
            self.feature_sets.append(cols)

            if eval_set is not None:
                val_pred = self._raw_predict(X_valp)
                score = np.mean((y_val - val_pred) ** 2)

                if self.verbose:
                    print(f"Round {m+1}: val_mse={score:.6f}")

                if score < best_score - 1e-8:
                    best_score = score
                    best_trees = deepcopy(self.trees)
                    best_feature_sets = deepcopy(self.feature_sets)
                    rounds_no_improve = 0
                else:
                    rounds_no_improve += 1
                    if self.early_stopping_rounds is not None and rounds_no_improve >= self.early_stopping_rounds:
                        if self.verbose:
                            print(f"Early stopping at round {m+1}")
                        break

        if eval_set is not None and best_trees:
            self.trees = best_trees
            self.feature_sets = best_feature_sets

        return self

    def _raw_predict(self, Xp):
        y_pred = np.full(Xp.shape[0], self.base_pred, dtype=np.float32)
        for tree, cols in zip(self.trees, self.feature_sets):
            y_pred += self.lr * tree.predict(Xp[:, cols]) / (1.0 + self.reg_lambda)
        return y_pred

    def predict(self, X):
        Xp = self._to_matrix(X, fit=False)
        return self._raw_predict(Xp)


Improved XGBoost (Custom) — What Was Changed & Why It Works Better
Overview
The original implementation was a basic gradient boosting regressor. However, the dataset contains:
Many categorical features
Significant missing values
Mixed data types
The updated version addresses these issues and introduces optimizations inspired by real XGBoost.
1. Data Preprocessing (Major Accuracy Boost)
Problem
Model expected numeric input
Dataset contains many object columns
Missing values were ignored
Solution
Numeric Features → Median Imputation
Categorical Features → Mode Imputation + One-Hot Encoding
Impact
Enables model to use all features correctly
Prevents loss of information due to missing values
Converts categorical data into usable numerical format
 Biggest improvement in overall accuracy
2. Column Subsampling (colsample_bytree)
What was added
Each tree trains on a random subset of features
 Why it helps
Reduces overfitting
Encourages diverse trees
Mimics real XGBoost behavior
 3. Row Subsampling (subsample)
 What was added
Each tree trains on a random subset of data
 Why it helps
Reduces variance
Prevents memorization of training data
Improves generalization
 4. Stronger Tree Learner
 Before
Custom tree (less optimized)
 After
Used DecisionTreeRegressor
 Why it helps
Better splits
Faster computation
More stable training
 5. Learning Rate + Shrinkage
 Before
Python
f_m += lr * preds
 After
Python
pred += lr * update / (1 + reg_lambda)
 Why it helps
Prevents large jumps
Stabilizes training
Adds regularization effect
 6. Regularization (reg_lambda)
 What it does
Penalizes large updates
 Why it helps
Reduces overfitting
Makes model more robust on noisy data
 7. Early Stopping
 What was added
Stops training if validation error stops improving
 Why it helps
Avoids over-training
Saves computation time
Improves real-world performance
 8. Feature-wise Training Tracking
What was added
Each tree stores:
Its feature subset
Its learned structure
Why it helps
Ensures correct prediction mapping
Prevents feature mismatch issues
Overall Improvements
Component
Effect
Data preprocessing
 Huge accuracy boost
Subsampling
Better generalization
Strong trees
Faster + more accurate
Regularization
Prevents overfitting
Early stopping
Efficient training

In [37]:
import os, time, warnings, pickle
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection   import train_test_split, StratifiedKFold
from sklearn.preprocessing     import LabelEncoder, RobustScaler, OrdinalEncoder
from sklearn.impute            import SimpleImputer, KNNImputer
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.linear_model      import LogisticRegression
from sklearn.metrics           import (accuracy_score, f1_score,
                                       balanced_accuracy_score,
                                       classification_report)
from sklearn.ensemble          import VotingClassifier

import lightgbm  as lgb
import xgboost   as xgb
import catboost  as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import subprocess
    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name",
                                        "--format=csv,noheader"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    HAS_GPU = True
    print(f"✓ GPU detected: {gpu_info}")
except Exception:
    HAS_GPU = False
    print("⚠ No GPU detected — falling back to CPU (set device flags below)")

LGBM_DEVICE  = "gpu"  if HAS_GPU else "cpu"
XGB_DEVICE   = "cuda" if HAS_GPU else "cpu"
CB_TASK_TYPE = "GPU"  if HAS_GPU else "CPU"

DATA_DIR = "/kaggle/input/datasets/shreyash28/pharam1"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
sub_df   = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

✓ GPU detected: Tesla T4
Tesla T4


In [38]:
print(f"\n✓ train : {train_df.shape}  |  test : {test_df.shape}")
test_ids = (test_df["Patient Id"].values
            if "Patient Id" in test_df.columns
            else np.arange(len(test_df)))

DROP = [
    "Patient Id", "Patient First Name", "Family Name", "Father's name",
    "Institute Name", "Test 1", "Test 2", "Test 3", "Test 5",
    "Parental consent", "Test 4", "Location of Institute",
]
def drop_safe(df, cols):
    return df.drop(columns=[c for c in cols if c in df.columns], errors="ignore")

train_df = drop_safe(train_df, DROP)
test_df  = drop_safe(test_df,  DROP)

train_df = train_df.dropna(subset=["Genetic Disorder", "Disorder Subclass"], how="all")
print(f"  Training rows after target-null drop: {len(train_df):,}")

le_gd, le_ds = LabelEncoder(), LabelEncoder()
mask_gd = train_df["Genetic Disorder"].notna()
mask_ds = train_df["Disorder Subclass"].notna()
y_gd_raw = train_df["Genetic Disorder"].copy()
y_ds_raw = train_df["Disorder Subclass"].copy()
le_gd.fit(train_df.loc[mask_gd, "Genetic Disorder"])
le_ds.fit(train_df.loc[mask_ds, "Disorder Subclass"])

print(f"\n  GD classes ({len(le_gd.classes_)}): {list(le_gd.classes_)}")
print(f"  DS classes ({len(le_ds.classes_)}): {list(le_ds.classes_)}")


✓ train : (22083, 45)  |  test : (9465, 43)
  Training rows after target-null drop: 21,805

  GD classes (3): ['Mitochondrial genetic inheritance disorders', 'Multifactorial genetic inheritance disorders', 'Single-gene inheritance diseases']
  DS classes (9): ["Alzheimer's", 'Cancer', 'Cystic fibrosis', 'Diabetes', 'Hemochromatosis', "Leber's hereditary optic neuropathy", 'Leigh syndrome', 'Mitochondrial myopathy', 'Tay-Sachs']


In [39]:
SUBCLASS_TO_GD = {
    "Leigh syndrome"                      : "Mitochondrial genetic inheritance disorders",
    "Leber's hereditary optic neuropathy" : "Mitochondrial genetic inheritance disorders",
    "Mitochondrial myopathy"              : "Mitochondrial genetic inheritance disorders",
    "Cystic fibrosis"                     : "Single-gene inheritance diseases",
    "Tay-Sachs"                           : "Single-gene inheritance diseases",
    "Hemochromatosis"                     : "Single-gene inheritance diseases",
    "Alzheimer's"                         : "Multifactorial genetic inheritance disorders",
    "Cancer"                              : "Multifactorial genetic inheritance disorders",
    "Diabetes"                            : "Multifactorial genetic inheritance disorders",
}
GD_TO_SUBCLASSES = {}
for sc, gd in SUBCLASS_TO_GD.items():
    GD_TO_SUBCLASSES.setdefault(gd, []).append(sc)

In [40]:
def engineer(df):
    df = df.copy()

    # ── Age features
    for p, col in [("Mother's age", "mother"), ("Father's age", "father")]:
        if p in df.columns and "Patient Age" in df.columns:
            df[f"{col}_age_at_birth"] = df[p] - df["Patient Age"]
    if "Mother's age" in df.columns and "Father's age" in df.columns:
        df["parent_age_diff"] = df["Father's age"] - df["Mother's age"]
        df["parent_age_sum"]  = df["Father's age"] + df["Mother's age"]
        df["parent_age_prod"] = df["Father's age"] * df["Mother's age"]

    # ── Symptom aggregates
    sym = [c for c in df.columns if c.startswith("Symptom")]
    if sym:
        df["symptom_count"]   = df[sym].sum(axis=1, min_count=1)
        df["symptom_ratio"]   = df["symptom_count"] / len(sym)
        # pairwise symptom interactions (top combos)
        for i in range(len(sym)):
            for j in range(i+1, len(sym)):
                df[f"sym_{i+1}x{j+1}"] = df[sym[i]].fillna(0) * df[sym[j]].fillna(0)

    # ── Gene risk
    gene_flag_cols = [
        "Genes in mother's side", "Inherited from father",
        "Maternal gene", "Paternal gene"
    ]
    gbins = []
    for col in gene_flag_cols:
        if col in df.columns:
            df[f"{col}_bin"] = (df[col] == "Yes").astype(float)
            gbins.append(f"{col}_bin")
    if gbins:
        df["gene_risk_score"]  = df[gbins].sum(axis=1)
        df["gene_risk_sq"]     = df["gene_risk_score"] ** 2
        m = df.get("Genes in mother's side_bin", pd.Series(0, index=df.index))
        f = df.get("Inherited from father_bin",  pd.Series(0, index=df.index))
        df["both_sides_gene"]  = ((m == 1) & (f == 1)).astype(float)

    # ── Blood & vitals
    if ("Blood cell count (mcL)" in df.columns and
            "White Blood cell count (thousand per microliter)" in df.columns):
        df["blood_cell_ratio"] = (
            df["Blood cell count (mcL)"] /
            df["White Blood cell count (thousand per microliter)"].replace(0, np.nan)
        )
        df["wbc_x_age"] = (
            df["White Blood cell count (thousand per microliter)"] *
            df.get("Patient Age", 0)
        )
    if "Respiratory Rate (breaths/min)" in df.columns:
        df["tachypnea_flag"]   = (df["Respiratory Rate (breaths/min)"] == "Tachypnea").astype(float)
    if "Heart Rate (rates/min" in df.columns:
        df["tachycardia_flag"] = (df["Heart Rate (rates/min"] == "Tachycardia").astype(float)

    # ── Environmental risk
    env_cols = [
        "H/O serious maternal illness", "H/O radiation exposure (x-ray)",
        "H/O substance abuse", "Assisted conception IVF/ART",
        "History of anomalies in previous pregnancies", "Birth asphyxia"
    ]
    ebins = []
    for col in env_cols:
        if col in df.columns:
            df[f"{col}_bin"] = (df[col] == "Yes").astype(float)
            ebins.append(f"{col}_bin")
    if ebins:
        df["env_risk_score"] = df[ebins].sum(axis=1)
        df["env_x_gene"]     = df["env_risk_score"] * df.get("gene_risk_score", 0)

    # ── Gender encoding
    if "Gender" in df.columns:
        df["is_male"] = (df["Gender"].str.lower().str.strip() == "male").astype(float)

    return df

In [41]:
train_df = engineer(train_df)
test_df  = engineer(test_df)

TARGET_COLS  = ["Genetic Disorder", "Disorder Subclass"]
feature_cols = [c for c in train_df.columns if c not in TARGET_COLS]

X_all  = train_df[feature_cols]
X_test = test_df.reindex(columns=feature_cols)   # fills any missing cols

num_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_all.select_dtypes(include=["object"]).columns.tolist()

print(f"  Numeric   : {len(num_cols)}")
print(f"  Categorical: {len(cat_cols)}")
print(f"  Total      : {len(num_cols)+len(cat_cols)}")


  Numeric   : 48
  Categorical: 20
  Total      : 68


In [42]:
def make_preprocessor():
    num_pipe = Pipeline([
        ("impute", KNNImputer(n_neighbors=5, weights="distance")),
        ("scale",  RobustScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value",
                                  unknown_value=-1,
                                  encoded_missing_value=-2)),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")

def smote(X_arr, y_arr, ratio=0.7, k=5, seed=42):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y_arr, return_counts=True)
    target = int(counts.max() * ratio)

    Xs, ys = [X_arr], [y_arr]
    for cls, cnt in zip(classes, counts):
        if cnt >= target:
            continue
        needed = target - cnt
        Xc = X_arr[y_arr == cls]
        Xf = np.where(np.isnan(Xc),
                      np.where(np.isnan(np.nanmean(Xc, 0)), 0, np.nanmean(Xc, 0)),
                      Xc)
        k_ = max(1, min(k, len(Xc)-1))
        syn = []
        for _ in range(needed):
            i   = rng.integers(len(Xc))
            s   = Xf[i]
            d   = np.linalg.norm(Xf - s, axis=1); d[i] = np.inf
            nn  = Xf[rng.choice(np.argsort(d)[:k_])]
            syn.append(s + rng.uniform(0,1)*(nn - s))
        Xs.append(np.array(syn))
        ys.append(np.full(needed, cls))

    X_out = np.vstack(Xs);  y_out = np.concatenate(ys)
    p = rng.permutation(len(y_out))
    return X_out[p], y_out[p]

In [43]:
def make_lgbm(n_classes, extra_params=None):
    params = dict(
        n_estimators    = 2000,
        learning_rate   = 0.03,
        num_leaves      = 127,
        max_depth       = -1,
        min_child_samples = 20,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        reg_alpha       = 0.1,
        reg_lambda      = 1.0,
        class_weight    = "balanced",
        device          = LGBM_DEVICE,
        random_state    = 42,
        n_jobs          = -1,
    )
    if extra_params:
        params.update(extra_params)
    return lgb.LGBMClassifier(**params)

In [44]:
# def make_xgb(n_classes, extra_params=None):
#     params = dict(
#         n_estimators        = 2000,
#         learning_rate       = 0.03,
#         max_depth           = 7,
#         subsample           = 0.8,
#         colsample_bytree    = 0.8,
#         reg_alpha           = 0.1,
#         reg_lambda          = 1.0,
#         use_label_encoder   = False,
#         eval_metric         = "mlogloss",
#         device              = XGB_DEVICE,
#         random_state        = 42,
#         verbose             = -1,
#         n_jobs              = -1,
#     )
#     if extra_params:
#         params.update(extra_params)
#     return XGBClassifier(**params)

In [45]:
def optuna_lgbm(X_tr, y_tr, n_classes, n_trials=40, seed=42):
    """Quick Optuna search on LightGBM hyperparams (CV on training fold only)."""
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)

    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int("n_estimators", 500, 3000, step=200),
            num_leaves        = trial.suggest_int("num_leaves", 31, 255),
            max_depth         = trial.suggest_int("max_depth", 4, 12),
            learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            subsample         = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree  = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_samples = trial.suggest_int("min_child_samples", 10, 60),
            reg_alpha         = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            reg_lambda        = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            class_weight      = "balanced",
            device            = LGBM_DEVICE,
            random_state      = seed,
            n_jobs            = -1,
            verbosity         = -1,
        )
        scores = []
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            m = lgb.LGBMClassifier(**params)
            m.fit(X_tr[tr_idx], y_tr[tr_idx],
                  eval_set=[(X_tr[val_idx], y_tr[val_idx])],
                  callbacks=[lgb.early_stopping(30, verbose=False),
                              lgb.log_evaluation(-1)])
            scores.append(f1_score(y_tr[val_idx],
                                   m.predict(X_tr[val_idx]),
                                   average="macro"))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f"     Optuna best F1-macro: {study.best_value:.4f}  "
          f"(trials: {len(study.trials)})")
    return study.best_params

In [46]:
def soft_vote(proba_list, weights=None):
    if weights is None:
        weights = [1.0] * len(proba_list)
    w = np.array(weights) / sum(weights)
    return sum(p * wi for p, wi in zip(proba_list, w))

In [47]:
def apply_hierarchy(gd_pred_labels, ds_pred_labels, ds_proba, le_ds):
    corrected = ds_pred_labels.copy()
    n_fixed   = 0
    for i, (gd, ds) in enumerate(zip(gd_pred_labels, ds_pred_labels)):
        allowed = GD_TO_SUBCLASSES.get(gd, [])
        if ds not in allowed and allowed:
            # pick highest-prob subclass that IS compatible
            allowed_idx = [j for j, cls in enumerate(le_ds.classes_)
                           if cls in allowed]
            if allowed_idx:
                best = allowed_idx[np.argmax(ds_proba[i, allowed_idx])]
                corrected[i] = le_ds.classes_[best]
                n_fixed += 1
    print(f"     Hierarchy post-processing: fixed {n_fixed}/{len(corrected)} predictions")
    return corrected


In [48]:
def run_task(task_name, mask, y_raw, le, X_extra_train=None,
             X_extra_test=None, run_optuna=True):
    print(f"  TASK : {task_name}")

    X_task = X_all[mask].copy()
    y_task = le.transform(y_raw[mask])
    n_cls  = len(le.classes_)

    vc   = pd.Series(y_task).value_counts()
    keep = pd.Series(y_task).isin(vc[vc >= 2].index).values
    if keep.sum() < len(y_task):
        print(f"  ⚠ Removed {len(y_task)-keep.sum()} singleton-class rows")
    X_task = X_task[keep]
    y_task = y_task[keep]
    if X_extra_train is not None:
        X_extra_train = X_extra_train[keep]

    print(f"  Dataset: {len(X_task):,} samples  |  {n_cls} classes")
    for i, cnt in pd.Series(y_task).value_counts().sort_index().items():
        print(f"    [{le.classes_[i]}]: {cnt:,}")

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_task, y_task, test_size=0.2,
        stratify=y_task, random_state=42
    )
    if X_extra_train is not None:
        mask_tr = np.isin(np.where(keep)[0],
                          np.where(keep)[0][
                              np.isin(np.arange(keep.sum()),
                                      np.where(
                                          pd.Series(np.arange(len(X_task))).isin(
                                              np.where(~np.isin(
                                                  np.arange(len(X_task)),
                                                  np.arange(int(len(X_task)*0.2))
                                              ))[0]
                                          ).values
                                      )[0]
                          )])
        
        xet_tr, xet_te = train_test_split(
            X_extra_train, test_size=0.2,
            stratify=y_task, random_state=42
        )

    print("\nPreprocessing")
    pre = make_preprocessor()
    X_tr_pp = pre.fit_transform(X_tr)
    X_te_pp = pre.transform(X_te)
    X_ts_pp = pre.transform(X_test.reindex(columns=feature_cols))

    if X_extra_train is not None:
        X_tr_pp = np.hstack([X_tr_pp, xet_tr])
        X_te_pp = np.hstack([X_te_pp, xet_te])
        X_ts_pp = np.hstack([X_ts_pp, X_extra_test])
        print(f"     + {xet_tr.shape[1]} hierarchical features appended")

    print("oversampling")
    X_tr_aug, y_tr_aug = smote(X_tr_pp, y_tr, ratio=0.7, k=5)
    print(f"     {len(y_tr):,} → {len(y_tr_aug):,} samples")

    lgbm_extra = {}
    if run_optuna:
        print(f"  → Optuna search (40 trials on LightGBM)...")
        t0 = time.time()
        best_p = optuna_lgbm(X_tr_aug, y_tr_aug, n_classes=n_cls, n_trials=40)
        lgbm_extra = {k: v for k, v in best_p.items()}
        print(f"     Search time: {time.time()-t0:.1f}s")

    print("Training XGBoost")
    t0 = time.time()
    lgbm_m = make_lgbm(n_cls, lgbm_extra)
    lgbm_m.fit(X_tr_aug, y_tr_aug,
               eval_set=[(X_te_pp, y_te)],
               callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(-1)])
    print(f"     Done in {time.time()-t0:.1f}s")

    # print("Training XGBoost")
    # t0 = time.time()
    # xgb_m = make_xgb(n_cls)
    # xgb_m.fit(X_tr_aug, y_tr_aug,
    #           eval_set=[(X_te_pp, y_te)],
    #           verbose=False)
    # print(f"     Done in {time.time()-t0:.1f}s")

    # print("  → Training CatBoost (GPU)...")
    # t0 = time.time()
    # cat_m = make_catboost(n_cls)
    # cat_m.fit(X_tr_aug, y_tr_aug,
    #           eval_set=(X_te_pp, y_te),
    #           verbose=0)
    # print(f"     Done in {time.time()-t0:.1f}s")

    lgbm_w,= (
        f1_score(y_te, lgbm_m.predict(X_te_pp), average="macro"),
        # f1_score(y_te, xgb_m.predict(X_te_pp),  average="macro"),
        # f1_score(y_te, cat_m.predict(X_te_pp),  average="macro"),
    )
    print(f"Individual val F1-macro → LGBM:{lgbm_w:.4f}")

    val_proba_list  = [lgbm_m.predict_proba(X_te_pp)]
    test_proba_list = [lgbm_m.predict_proba(X_ts_pp)]

    val_proba  = soft_vote(val_proba_list,  [lgbm_w])
    test_proba = soft_vote(test_proba_list, [lgbm_w])

    y_pred     = np.argmax(val_proba, axis=1)
    test_preds = np.argmax(test_proba, axis=1)

    acc     = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    f1_mac  = f1_score(y_te, y_pred, average="macro")
    f1_w    = f1_score(y_te, y_pred, average="weighted")

    print(f"\Ensemble Results")
    print(f"  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Balanced Accuracy : {bal_acc:.4f}  ({bal_acc*100:.2f}%)")
    print(f"  F1 Macro          : {f1_mac:.4f}")
    print(f"  F1 Weighted       : {f1_w:.4f}")
    present_labels = np.unique(y_te)
    print(f"Classification Report:")
    print(classification_report(y_te, y_pred,
                                target_names=le.classes_[present_labels],
                                labels=present_labels,
                                zero_division=0))

    fi = lgbm_m.feature_importances_
    fn = (num_cols + cat_cols)[:len(fi)]
    top = sorted(zip(fn, fi), key=lambda x: -x[1])[:15]
    print(" Top 15 Features (LightGBM gain):")
    for name, imp in top:
        bar = "" * int(imp / max(fi) * 30)
        print(f"    {name:<45} {imp:7.0f}  {bar}")

    test_pred_labels = le.inverse_transform(test_preds)
    return (lgbm_m, pre,
            acc, bal_acc, f1_mac,
            test_proba, test_pred_labels,
            val_proba, y_te)

In [49]:
print("─Pre-processing test set")
pre_shared = make_preprocessor()
pre_shared.fit(X_all)
X_test_pp = pre_shared.transform(X_test)
print(f" Test preprocessed: {X_test_pp.shape}")

─Pre-processing test set
 Test preprocessed: (9465, 68)


In [52]:
import time
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

print("─Pre-processing test set")
pre_shared = make_preprocessor()
pre_shared.fit(X_all)
X_test_pp = pre_shared.transform(X_test)
print(f" Test preprocessed: {X_test_pp.shape}")

def run_task(task_name, mask, y_raw, le, X_extra_train=None,
             X_extra_test=None, run_optuna=True):
    print(f"  TASK : {task_name}")

    X_task = X_all[mask].copy()
    y_task = le.transform(y_raw[mask])
    n_cls  = len(le.classes_)

    vc   = pd.Series(y_task).value_counts()
    keep = pd.Series(y_task).isin(vc[vc >= 2].index).values
    if keep.sum() < len(y_task):
        print(f"  ⚠ Removed {len(y_task)-keep.sum()} singleton-class rows")
    X_task = X_task[keep]
    y_task = y_task[keep]
    if X_extra_train is not None:
        X_extra_train = X_extra_train[keep]

    print(f"  Dataset: {len(X_task):,} samples  |  {n_cls} classes")
    for i, cnt in pd.Series(y_task).value_counts().sort_index().items():
        print(f"    [{le.classes_[i]}]: {cnt:,}")

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_task, y_task, test_size=0.2,
        stratify=y_task, random_state=42
    )

    if X_extra_train is not None:
        xet_tr, xet_te = train_test_split(
            X_extra_train, test_size=0.2,
            stratify=y_task, random_state=42
        )

    print("\nPreprocessing")
    pre = make_preprocessor()
    X_tr_pp = pre.fit_transform(X_tr)
    X_te_pp = pre.transform(X_te)
    X_ts_pp = pre.transform(X_test.reindex(columns=feature_cols))

    if X_extra_train is not None:
        X_tr_pp = np.hstack([X_tr_pp, xet_tr])
        X_te_pp = np.hstack([X_te_pp, xet_te])
        X_ts_pp = np.hstack([X_ts_pp, X_extra_test])
        print(f"     + {xet_tr.shape[1]} hierarchical features appended")

    print("oversampling")
    X_tr_aug, y_tr_aug = smote(X_tr_pp, y_tr, ratio=0.7, k=5)
    print(f"     {len(y_tr):,} → {len(y_tr_aug):,} samples")

    lgbm_extra = {}
    if run_optuna:
        print(f"  → Optuna search (40 trials on LightGBM)...")
        t0 = time.time()
        best_p = optuna_lgbm(X_tr_aug, y_tr_aug, n_classes=n_cls, n_trials=40)
        lgbm_extra = {k: v for k, v in best_p.items()}
        print(f"     Search time: {time.time()-t0:.1f}s")

    print("Training XGBoost")
    t0 = time.time()
    lgbm_m = make_lgbm(n_cls, lgbm_extra)
    lgbm_m.fit(
        X_tr_aug, y_tr_aug,
        eval_set=[(X_te_pp, y_te)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(-1)]
    )
    print(f"     Done in {time.time()-t0:.1f}s")

    model_filename = f"/kaggle/working/xgboost_{task_name.replace(' ', '_').lower()}.pkl"

    joblib.dump(lgbm_m, model_filename)
    
    print(f"     Saved XGBoost model to: {model_filename}")

    lgbm_w = f1_score(y_te, lgbm_m.predict(X_te_pp), average="macro")
    print(f"Individual val F1-macro → XGBoost:{lgbm_w:.4f}")

    val_proba_list  = [lgbm_m.predict_proba(X_te_pp)]
    test_proba_list = [lgbm_m.predict_proba(X_ts_pp)]

    val_proba  = soft_vote(val_proba_list,  [lgbm_w])
    test_proba = soft_vote(test_proba_list, [lgbm_w])

    y_pred     = np.argmax(val_proba, axis=1)
    test_preds = np.argmax(test_proba, axis=1)

    acc     = accuracy_score(y_te, y_pred)
    bal_acc = balanced_accuracy_score(y_te, y_pred)
    f1_mac  = f1_score(y_te, y_pred, average="macro")
    f1_w    = f1_score(y_te, y_pred, average="weighted")

    print(f"  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Balanced Accuracy : {bal_acc:.4f}  ({bal_acc*100:.2f}%)")
    print(f"  F1 Macro          : {f1_mac:.4f}")
    print(f"  F1 Weighted       : {f1_w:.4f}")
    present_labels = np.unique(y_te)
    print(f"Classification Report:")
    print(classification_report(
        y_te, y_pred,
        target_names=le.classes_[present_labels],
        labels=present_labels,
        zero_division=0
    ))

    fi = lgbm_m.feature_importances_
    fn = (num_cols + cat_cols)[:len(fi)]
    top = sorted(zip(fn, fi), key=lambda x: -x[1])[:15]
    print(" Top 15 Features (LightGBM gain):")
    for name, imp in top:
        bar = "" * int(imp / max(fi) * 30)
        print(f"    {name:<45} {imp:7.0f}  {bar}")

    test_pred_labels = le.inverse_transform(test_preds)
    return (lgbm_m, pre,
            acc, bal_acc, f1_mac,
            test_proba, test_pred_labels,
            val_proba, y_te)

(pre_gd,
 acc_gd, bal_gd, f1_gd,
 test_proba_gd, preds_gd,
 val_proba_gd, y_te_gd) = run_task(
    "Genetic Disorder (3 classes)",
    mask_gd, y_gd_raw, le_gd,
    X_extra_train=None,
    X_extra_test=None,
    run_optuna=True,
)

─Pre-processing test set
 Test preprocessed: (9465, 68)
  TASK : Genetic Disorder (3 classes)
  Dataset: 19,937 samples  |  3 classes
    [Mitochondrial genetic inheritance disorders]: 10,202
    [Multifactorial genetic inheritance disorders]: 2,071
    [Single-gene inheritance diseases]: 7,664

Preprocessing
oversampling
     15,949 → 20,004 samples
  → Optuna search (40 trials on LightGBM)...
     Optuna best F1-macro: 0.6897  (trials: 40)
     Search time: 1706.4s
Training XGBoost
     Done in 10.6s
     Saved XGBoost model to: /kaggle/working/xgboost_genetic_disorder_(3_classes).pkl
Individual val F1-macro → XGBoost:0.5566
  Accuracy          : 0.6146  (61.46%)
  Balanced Accuracy : 0.5545  (55.45%)
  F1 Macro          : 0.5566
  F1 Weighted       : 0.6124
Classification Report:
                                              precision    recall  f1-score   support

 Mitochondrial genetic inheritance disorders       0.69      0.72      0.70      2041
Multifactorial genetic inheritanc

ValueError: too many values to unpack (expected 8)